# Opion Chore 2 — The Locked-Drawer Quiz (test-partition evaluation)

**Grades both trained models on data they have NEVER seen** — the 4% test
partition that was locked at dataset build time and never opened (not for
training, not for checkpoint selection).

## How to use
1. **Runtime → Change runtime type → T4 GPU** (do this first!)
2. Run cells top-to-bottom:
   - Cell 3 downloads the packed dataset (same 480MB tar as training, via gdown)
   - Cell 4 asks you to **drag-drop `final_act.pt` + `final_disc.pt`** into the
     file panel on the left (3.3MB each — the models you downloaded after training)
3. Pure inference — no training, ~5 minutes total.
4. Cell 10 prints **THE_REPORT** (JSON) — paste it back into the chat.

Rules of the quiz: the models answer every test question cold (no learning),
and the grades are broken down per street so we can see WHERE the student is
weak (that's the targeting map for RL). This drawer gets opened once for the
distillation stage — after this, it stays locked.

In [ ]:
# 1. Runtime check — must be T4 GPU
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU — set Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Keepalive: free -h every 5 min in the background
import threading, time, subprocess

def _ka():
    while True:
        try:
            out = subprocess.run(["free", "-h"], capture_output=True, text=True).stdout
            print("[keepalive]\n" + out, flush=True)
        except Exception as e:
            print("[keepalive] err", e, flush=True)
        time.sleep(300)

threading.Thread(target=_ka, daemon=True).start()
print("keepalive running (free -h every 300s)")

In [ ]:
# 3. Get the packed dataset (40 shards): reuse session copy if present
import glob, os, tarfile

if not glob.glob("packed/*/*/acts.npz"):
    if not os.path.exists("packed.tar"):
        import gdown
        gdown.download(id="1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA", output="packed.tar")
    with tarfile.open("packed.tar") as tf:
        tf.extractall("/content")
shards = sorted(glob.glob("packed/*/*/acts.npz"))
assert len(shards) == 40, f"expected 40 shards, found {len(shards)}"
print(len(shards), "shards found")

In [ ]:
# 4. Upload the two trained models
#    Drag-drop final_act.pt + final_disc.pt into the file panel (left),
#    or pick them in the dialog this cell opens.
from google.colab import files
import os

missing = [n for n in ("final_act.pt", "final_disc.pt") if not os.path.exists(n)]
if missing:
    print("upload:", missing)
    files.upload()
still = [n for n in ("final_act.pt", "final_disc.pt") if not os.path.exists(n)]
assert not still, f"still missing: {still} — re-run this cell"
print("models present:", sorted(f for f in os.listdir('.') if f.endswith('.pt')))

In [ ]:
# 5. Model definitions (byte-exact copies of the training architectures)
import torch
import torch.nn as nn

N_TOK, N_CTX, N_ACT, VOCAB = 41, 17, 6, 85


def _map_tokens(t):
    """cards stay 0..52; history tokens 1..32 shift to 53..84."""
    t = t.long()
    return torch.where(t < 53, t, 52 + t.clamp(min=1, max=32))


class Block(nn.Module):
    def __init__(self, d, h, f):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.h = h
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, f)
        self.fc2 = nn.Linear(f, d)
        self.act = nn.GELU()

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q = q.view(B, T, self.h, -1).transpose(1, 2)
        k = k.view(B, T, self.h, -1).transpose(1, 2)
        v = v.view(B, T, self.h, -1).transpose(1, 2)
        att = torch.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), -1)
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, D))
        x = x + self.fc2(self.act(self.fc1(self.ln2(x))))
        return x


class OpionPolicy(nn.Module):
    def __init__(self, d=128, layers=4, heads=4, f=512):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(N_TOK, d)
        self.ctx_proj = nn.Sequential(nn.LayerNorm(N_CTX), nn.Linear(N_CTX, d))
        self.blocks = nn.ModuleList(Block(d, heads, f) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, N_ACT)

    def forward(self, tokens, ctx):
        x = self.emb(_map_tokens(tokens)) + self.pos.weight[None, :N_TOK]
        x = x + self.ctx_proj(ctx.float())[:, None, :]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln_f(x).mean(dim=1))

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


N_TOK_D, N_CTX_D = 8, 16


class DiscPolicy(nn.Module):
    def __init__(self, d=128, layers=4, heads=4, f=512):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(N_TOK_D, d)
        self.ctx_proj = nn.Sequential(nn.LayerNorm(N_CTX_D), nn.Linear(N_CTX_D, d))
        self.blocks = nn.ModuleList(Block(d, heads, f) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, 3)

    def forward(self, tokens, ctx):
        x = self.emb(tokens.long()) + self.pos.weight[None, :N_TOK_D]
        x = x + self.ctx_proj(ctx.float())[:, None, :]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln_f(x).mean(dim=1))

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


dev = "cuda"
m_act = OpionPolicy()
m_act.load_state_dict(torch.load("final_act.pt", map_location="cpu")["model"])
m_act.to(dev).eval()
m_disc = DiscPolicy()
m_disc.load_state_dict(torch.load("final_disc.pt", map_location="cpu")["model"])
m_disc.to(dev).eval()
print("act params:", f"{m_act.n_params():,}", "| disc params:", f"{m_disc.n_params():,}")
assert m_act.n_params() == 812584, "act arch mismatch!"
assert m_disc.n_params() == 807843, "disc arch mismatch!"
print("MODELS_LOADED")

In [ ]:
# 6. Open the LOCKED DRAWER: part==2 (test) — never trained on, never peeked.
#    Also load val (part==1, capped 200k like training) for the side-by-side.
import glob
import numpy as np
import torch

ACT_KEYS = ("tokens", "ctx", "legal", "p_soft", "act_idx", "street")


def load_act(part, cap=0):
    cols = {k: [] for k in ACT_KEYS}
    for f in sorted(glob.glob("packed/*/*/acts.npz")):
        z = np.load(f, allow_pickle=False)
        sel = z["part"] == part
        if sel.any():
            for k in ACT_KEYS:
                cols[k].append(z[k][sel])
    out = {k: np.concatenate(v) for k, v in cols.items()}
    if cap and len(out["act_idx"]) > cap:
        out = {k: v[:cap] for k, v in out.items()}
    return out


def to_gpu(d):
    ints = ("tokens", "act_idx", "street", "target")
    return {k: torch.from_numpy(v.astype(np.int64 if k in ints else np.float32)).to("cuda")
            for k, v in d.items()}


act_test = to_gpu(load_act(2))
act_val = to_gpu(load_act(1, cap=200_000))
print(f"act: test={len(act_test['act_idx']):,} val={len(act_val['act_idx']):,}")

DISC_KEYS = ("tokens", "ctx", "target", "street")


def load_disc(part):
    cols = {k: [] for k in DISC_KEYS}
    for f in sorted(glob.glob("packed/*/*/disc.npz")):
        z = np.load(f, allow_pickle=False)
        sel = z["part"] == part
        if sel.any():
            for k in DISC_KEYS:
                cols[k].append(z[k][sel])
    return {k: np.concatenate(v) for k, v in cols.items()}


disc_test = to_gpu(load_disc(2))
print(f"disc test: {len(disc_test['target']):,}")

# train-split target distribution -> majority-class baseline
cnt = np.zeros(3, dtype=np.int64)
for f in sorted(glob.glob("packed/*/*/disc.npz")):
    z = np.load(f, allow_pickle=False)
    t = z["target"][z["part"] == 0]
    cnt += np.bincount(t, minlength=3)
MAJ = int(cnt.argmax())
print("disc train target counts:", cnt.tolist(), "-> majority card index", MAJ)
print("DATA_LOADED")

In [ ]:
# 7. Eval machinery — fp32 everywhere (the report card is exact, no autocast)
BUCKETS = ["fold", "check/call", "small-raise", "mid-raise", "big-raise", "all-in"]
STREETS = ["preflop", "flop", "turn", "river"]


@torch.no_grad()
def act_q(model, d, batch=32768):
    qs = []
    for i in range(0, len(d["act_idx"]), batch):
        sl = slice(i, i + batch)
        logits = model(d["tokens"][sl], d["ctx"][sl]).float()
        logits = logits.masked_fill(d["legal"][sl] == 0, float("-inf"))
        qs.append(torch.softmax(logits, -1))
    return torch.cat(qs)


def act_metrics(p, q, act_idx, mask=None):
    if mask is not None:
        p, q, act_idx = p[mask], q[mask], act_idx[mask]
    kl = (p * (torch.log(p + 1e-8) - torch.log(q + 1e-8))).sum(-1).mean().item()
    tv = (0.5 * (p - q).abs().sum(-1)).mean().item()
    ag = (p.argmax(-1) == q.argmax(-1)).float().mean().item()
    ah = (q.argmax(-1) == act_idx).float().mean().item()
    return {"kl": round(kl, 4), "tv": round(tv, 4),
            "argmax_ag": round(ag, 4), "hard_ag": round(ah, 4), "n": int(len(p))}


print("eval fns ready")

In [ ]:
# 8. ACT MODEL report card: val vs test side-by-side, then per-street on test
p_v, p_t = act_val["p_soft"], act_test["p_soft"]
q_v, q_t = act_q(m_act, act_val), act_q(m_act, act_test)

val_m = act_metrics(p_v, q_v, act_val["act_idx"])
test_m = act_metrics(p_t, q_t, act_test["act_idx"])
print(f"{'partition':<10}{'KL':>8}{'TV':>8}{'argmax_ag':>11}{'hard_ag':>9}{'n':>11}")
for name, m in (("VAL", val_m), ("TEST", test_m)):
    print(f"{name:<10}{m['kl']:>8.4f}{m['tv']:>8.4f}{m['argmax_ag']:>11.4f}"
          f"{m['hard_ag']:>9.4f}{m['n']:>11,}")

print("\nper-street (TEST):")
by_street = {}
for s in range(4):
    m = act_metrics(p_t, q_t, act_test["act_idx"], mask=(act_test["street"] == s))
    by_street[STREETS[s]] = m
    print(f"  {STREETS[s]:<9}{m['kl']:>8.4f}{m['tv']:>8.4f}{m['argmax_ag']:>11.4f}"
          f"{m['hard_ag']:>9.4f}{m['n']:>11,}")

print("\naction mix on TEST (teacher hard action vs student argmax):")
mix_t = torch.bincount(act_test["act_idx"], minlength=6).float() / len(act_test["act_idx"])
mix_s = torch.bincount(q_t.argmax(-1), minlength=6).float() / len(q_t)
for b in range(6):
    print(f"  {BUCKETS[b]:<12} teacher {100 * mix_t[b]:5.2f}%   student {100 * mix_s[b]:5.2f}%")

In [ ]:
# 9. DISCARD MODEL report card: test accuracy vs two dumb baselines
import numpy as np

tgt = disc_test["target"]
tok = disc_test["tokens"]


@torch.no_grad()
def disc_q(model, d, batch=65536):
    outs = []
    for i in range(0, len(d["target"]), batch):
        sl = slice(i, i + batch)
        outs.append(model(d["tokens"][sl], d["ctx"][sl]).float())
    return torch.cat(outs)


q_d = disc_q(m_disc, disc_test)
pred = q_d.argmax(-1)
acc = (pred == tgt).float().mean().item()

# baseline 1: always discard the train-majority card index
acc_maj = float((MAJ == tgt.cpu().numpy()).mean())

# baseline 2: discard the lowest-rank hole card (dumb equity proxy:
# rank-major, card-id tie-break)
h = tok[:, :3].cpu().numpy().astype(np.int64)   # 3 hole cards (pre-discard)
key = ((h - 1) % 13) * 53 + h
acc_low = float((key.argmin(axis=1) == tgt.cpu().numpy()).mean())

print(f"DISC TEST: n={len(tgt):,}")
print(f"  student           {100 * acc:6.2f}%")
print(f"  majority (train)  {100 * acc_maj:6.2f}%   (always discard card index {MAJ})")
print(f"  lowest-rank card  {100 * acc_low:6.2f}%")

st = torch.bincount(disc_test["street"], minlength=4).tolist()
print("  test streets (0..3):", st, "(discards are flop-only by game rule)")
for s in range(4):
    m_ = disc_test["street"] == s
    if m_.any():
        a = (pred[m_] == tgt[m_]).float().mean().item()
        print(f"    street {STREETS[s]:<9} acc={a:.4f} n={int(m_.sum()):,}")

DISC = {"test_acc": round(acc, 4), "baseline_majority": round(acc_maj, 4),
        "baseline_lowcard": round(acc_low, 4), "n": int(len(tgt))}

In [ ]:
# 10. THE_REPORT — paste this JSON back into the chat
import json

THE_REPORT = {
    "act": {"val": val_m, "test": test_m, "test_by_street": by_street,
            "action_mix_test": {"teacher": [round(float(x), 4) for x in mix_t],
                                "student": [round(float(x), 4) for x in mix_s]}},
    "disc": DISC,
    "note": ("fp32 eval, no autocast; VAL here is freshly recomputed "
             "(cap 200k) so the val-vs-test comparison is apples-to-apples"),
}
print(json.dumps(THE_REPORT, indent=1))
print("QUIZ_DONE")